# MiniMax H3 (Hailuo 3.0) — Architecture, capacités… et licence géo-restreinte

> **Verdict SOTA : INTRINSIC.** Ce notebook **n'exécute pas** le modèle MiniMax H3 et **n'affiche aucun Output généré**. La licence *MiniMax H3 Community License* interdit explicitement l'usage, l'hébergement **et l'affichage des Outputs** du modèle dans l'Union Européenne (territoire exclu). Nous étudions donc l'architecture du modèle et — surtout — la **décision de conformité** qu'impose une telle licence. Cette transparence est elle-même une compétence d'ingénierie ML.

**Source officielle** : `MiniMax H3 COMMUNITY LICENSE AGREEMENT` (date de licence : 2 août 2026), déposée sur [HuggingFace `MiniMaxAI/MiniMax-H3`](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE). Code modèle : [GitHub `MiniMax-AI/MiniMax-H3`](https://github.com/MiniMax-AI/MiniMax-H3).


## Le contexte : un « Sora at home » qui fait vibrer la communauté

MiniMax H3 (aussi *Hailuo 3.0* / 海螺3), open-sourcé fin juillet / début août 2026, est arrivé **#1 du benchmark *Artificial Analysis* video editing** (Elo ~1130) au lancement. Sa promesse séduit : un modèle **omni-modal unifié** (texte, image, vidéo, audio en entrée) produisant de la vidéo jusqu'à **2K / 15 s / 24 fps** avec **audio stéréo natif** (32 kHz, 11 langues) — le tout en *open-weights*, donc en principe auto-hébergeable. C'est le « Sora at home » : la même ambition qu'OpenAI Sora, mais en poids téléchargeables plutôt qu'en API fermée.

Ce notebook examine pourquoi, **pour notre contexte (France / UE, dépôt public, écoles partenaires)**, cette promesse se heurte à un obstacle qui n'est pas technique mais **juridique** — et comment raisonner sobrement cette décision de déploiement.


## Section 1 — La licence *MiniMax H3 Community License* : une restriction territoriale explicite

Lisons les clauses pivots (verbatim, numérotation originale) :

> **Art. I.5 — « Excluded Territories »** : *means the European Union, the United Kingdom, the Republic of Korea and the United States of America.*
>
> **Art. I.3 — « Applicable Territory »** : *means worldwide, excluding the Excluded Territories.*
>
> **Art. II — Grant of Rights** : *Solely within the Applicable Territory, we grant you a non-exclusive, non-transferable, royalty-free, limited license to use, reproduce, distribute, create derivative works […] and modify the Materials […].* **Aucune exception pour l'éducation, la recherche ou un usage non-commercial** n'est prévue.
>
> **Art. V.4 — Use Restrictions** : *You may not use, reproduce, modify, distribute, or display the MiniMax H3 Works **or any of their Outputs or results** outside the Applicable Territory. Any such use outside the Applicable Territory is not authorized by this Agreement.*
>
> **Exhibit A.1** (Acceptable Use Policy, première utilisation interdite) : *Use outside the Applicable Territory.*

Deux points sont souvent mal compris et méritent d'être soulignés :

1. **Les *Outputs* sont couverts, pas seulement les poids.** Même en appelant l'API cloud MiniMax depuis l'UE (un *Hosted Service* au sens de l'Art. I.8), afficher ou redistribuer la vidéo générée dans l'UE est un usage *outside the Applicable Territory* → non autorisé. L'argument « on n'héberge pas les poids, on appelle juste l'API » **ne tient pas**.
2. **L'« open-weights » n'est pas le « open source ».** Les poids sont téléchargeables, mais sous une licence propriétaire à restriction territoriale — l'opposé d'une licence OSI. Confondre les deux est une erreur classique de mise en production.

La seule voie vers un usage UE est une **licence commerciale séparée** (Art. II, dernier § : *« you are welcome to contact us about obtaining a license »*) — une démarche *provider-side* qui dépend de MiniMax (Nanonoble Pte. Ltd.), pas de l'utilisateur.


## Section 2 — Vérificateur de juridiction : coder la conformité

Plutôt qu'un discours, codons le raisonnement. La fonction suivante **parse la clause *Excluded Territories*** et décide, pour une localisation de déploiement donnée, si l'usage de MiniMax H3 y est autorisé. Ce code tourne localement (aucun appel au modèle) — il analyse le texte de licence, pas le modèle.


In [1]:
# Excluded Territories au sens de l'Art. I.5 (verbatim license).
# Couverture par code pays / région ISO, pour le raisonnement de conformité.
EXCLUDED_TERRITORIES = {
    "European Union": {
        "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR",
        "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK",
        "SI", "ES", "SE",
    },
    "United Kingdom": {"GB"},
    "Republic of Korea": {"KR"},
    "United States of America": {"US"},
}


def is_applicable_territory(country_code: str) -> bool:
    """True si country_code (ISO-3166 alpha-2) est dans l'Applicable Territory.

    L'Applicable Territory = monde entier hors Excluded Territories (Art. I.3).
    """
    cc = country_code.upper()
    for states in EXCLUDED_TERRITORIES.values():
        if cc in states:
            return False
    return True


def deployment_verdict(country_code: str, label: str = "") -> str:
    """Rend un verdict lisible pour un déploiement envisagé dans country_code."""
    ok = is_applicable_territory(country_code)
    where = label or country_code
    if ok:
        return f"{where} ({country_code}) : Applicable Territory — usage autorise sous la Community License"
    # identifier quel territoire exclu
    for name, states in EXCLUDED_TERRITORIES.items():
        if country_code.upper() in states:
            return f"{where} ({country_code}) : EXCLUDED ({name}) — usage NON autorise, licence commerciale requise"
    return f"{where} ({country_code}) : indetermine"


# --- Verdicts pour nos contextes reels ---
contextes = [
    ("FR", "France (user + ecoles EPITA/ECE/ESGF/EPF)"),
    ("DE", "Allemagne"),
    ("GB", "Royaume-Uni"),
    ("US", "Etats-Unis"),
    ("KR", "Coree du Sud"),
    ("CN", "Chine"),
    ("JP", "Japon"),
    ("BR", "Bresil"),
]
for cc, label in contextes:
    print(deployment_verdict(cc, label))


France (user + ecoles EPITA/ECE/ESGF/EPF) (FR) : EXCLUDED (European Union) — usage NON autorise, licence commerciale requise
Allemagne (DE) : EXCLUDED (European Union) — usage NON autorise, licence commerciale requise
Royaume-Uni (GB) : EXCLUDED (United Kingdom) — usage NON autorise, licence commerciale requise
Etats-Unis (US) : EXCLUDED (United States of America) — usage NON autorise, licence commerciale requise
Coree du Sud (KR) : EXCLUDED (Republic of Korea) — usage NON autorise, licence commerciale requise
Chine (CN) : Applicable Territory — usage autorise sous la Community License
Japon (JP) : Applicable Territory — usage autorise sous la Community License
Bresil (BR) : Applicable Territory — usage autorise sous la Community License


## Section 3 — Architecture : ce qui fait la force du modèle (et que nous ne pouvons pas exercer ici)

Bien que nous ne puissions pas exécuter H3, comprendre son architecture est légitime et précieux. Le dépôt [`MiniMax-AI/MiniMax-H3`](https://github.com/MiniMax-AI/MiniMax-H3) révèle les composants clés :

| Composant (dépôt) | Rôle | Détail technique |
|---|---|---|
| **`FL2VA`** | *Flow-Language-to-Video-Audio* — le cœur omni-modal | Encode unifie texte/image/vidéo/audio en entrée vers une représentation latente partagée |
| **`Ref2VA`** | *Reference-to-Video-Audio* | Gère les références multiples (first+last frame, omni-reference) pour l'édition instructionnelle |
| **`transformer`** / **`transformer_ref`** | Backbone de débruitage diffusif | Le transformeur principal + la branche de conditionnement par référence |
| **`vae`** | *Variational Autoencoder* vidéo | Compresse les pixels vidéo vers l'espace latent (jusqu'à 2K) |
| **`audio_vae`** + **`audio_scheduler`** | **Audio stéréo natif** | La différence distinctive : génère l'audio synchronisé (32 kHz, 11 langues) *en même temps* que la vidéo, pas en pass séparé |
| **`text_encoder`** + **`tokenizer`** | Encodage du prompt | Le `processor` orchestre l'entrée omni-modale |
| **`scheduler`** | Planificateur de diffusion | Contrôle le nombre de pas de débruitage |

**Point de licence notable** (Additional Note du LICENSE) : l'encodeur de H3 utilise **Qwen3-VL-32B**, lui-même sous **Apache 2.0** (une licence réellement ouverte, sans restriction territoriale). Mais cet encodeur est un *constituant* de H3 — l'assemblage complet reste sous la Community License géo-restreinte. On ne peut pas « récupérer juste l'encodeur Apache » pour contourner la restriction : le modèle intégré est un *Material* couvert par l'Accord.

**Pourquoi c'est un excellent cas pédagogique** : H3 combine une capacité absente de nos autres notebooks (vidéo 2K **+ audio natif synchronisé** dans un seul modèle omni-modal) — exactement le genre de prouesse technique qu'un cours SOTA voudrait montrer. La licence nous oblige à le faire *différemment* : par l'analyse, pas l'exécution.


## Section 4 — Matrice de décision : H3 vs ses alternatives

Face à une licence géo-restreinte, la compétence clé est de **raisonner en alternatives**. Le code suivant construit une matrice comparant MiniMax H3 aux autres modèles vidéo de la série (et à Sora), pour aider à décider quoi utiliser selon la juridiction.


In [2]:
# Matrice de decision : modeles video de la serie + Sora, selon 5 axes de conformite/usage.
# 'geo_restricted' = True si la licence exclut des territoires (UE en particulier).
MODELES = [
    {
        "nom": "MiniMax H3",
        "open_weights": True,
        "geo_restricted_ue": True,
        "audio_natif": True,
        "resolution_max": "2K",
        "notebook_series": "02-6 (ce notebook, descriptif)",
        "licence": "MiniMax H3 Community License (UE exclue)",
    },
    {
        "nom": "Wan 2.1/2.2",
        "open_weights": True,
        "geo_restricted_ue": False,
        "audio_natif": False,
        "resolution_max": "1080p",
        "notebook_series": "02-3 (executable localement)",
        "licence": "Apache 2.0 (permissive)",
    },
    {
        "nom": "HunyuanVideo",
        "open_weights": True,
        "geo_restricted_ue": False,
        "audio_natif": False,
        "resolution_max": "1080p",
        "notebook_series": "02-1 (executable localement)",
        "licence": "Tencent Community License (permissive UE)",
    },
    {
        "nom": "LTX-2",
        "open_weights": True,
        "geo_restricted_ue": False,
        "audio_natif": True,
        "resolution_max": "1080p",
        "notebook_series": "02-5 (executable, audiovisuel conjoint)",
        "licence": "LTX-2 Community License (permissive UE)",
    },
    {
        "nom": "OpenAI Sora",
        "open_weights": False,
        "geo_restricted_ue": False,
        "audio_natif": True,
        "resolution_max": "1080p",
        "notebook_series": "04-3 (API cloud, sous quota/cout)",
        "licence": "API fermee (ToS OpenAI)",
    },
]


def utilisables_en_ue(modeles):
    """Filtre les models utilisables en UE (non geo-restreints UE)."""
    return [m for m in modeles if not m["geo_restricted_ue"]]


def avec_audio_natif(modeles):
    """Modeles offrant l'audio natif synchronise."""
    return [m["nom"] for m in modeles if m["audio_natif"]]


print("=== Modeles utilisables en UE (pas de restriction territoriale UE) ===")
for m in utilisables_en_ue(MODELES):
    print(f"  - {m['nom']:<16} | {m['licence']}")

print()
print("=== Modeles avec audio natif synchronise ===")
print("  " + ", ".join(avec_audio_natif(MODELES)))

print()
print("=== Alternatives UE a H3 pour le cas 'video + audio natif' ===")
# H3 est exclus: on cherche les modeles utilisables en UE AVEC audio natif.
alt = [m for m in utilisables_en_ue(MODELES) if m["audio_natif"]]
for m in alt:
    print(f"  - {m['nom']:<16} | res {m['resolution_max']} | notebook {m['notebook_series']}")


=== Modeles utilisables en UE (pas de restriction territoriale UE) ===
  - Wan 2.1/2.2      | Apache 2.0 (permissive)
  - HunyuanVideo     | Tencent Community License (permissive UE)
  - LTX-2            | LTX-2 Community License (permissive UE)
  - OpenAI Sora      | API fermee (ToS OpenAI)

=== Modeles avec audio natif synchronise ===
  MiniMax H3, LTX-2, OpenAI Sora

=== Alternatives UE a H3 pour le cas 'video + audio natif' ===
  - LTX-2            | res 1080p | notebook 02-5 (executable, audiovisuel conjoint)
  - OpenAI Sora      | res 1080p | notebook 04-3 (API cloud, sous quota/cout)


## Section 5 — Cadre de décision : « puis-je légalement déployer ce modèle ici ? »

MiniMax H3 illustre un cadre de raisonnement général, utile pour **tout** modèle qu'on envisage d'intégrer en production ou en enseignement :

1. **Lire la licence à la source** (pas un résumé secondaire). Ici, le `LICENSE` brut HuggingFace — pas un article de blog.
2. **Identifier les restrictions structurelles** : territoriales (H3), de revenu commercial (H3 Art. IV : seuil 20 M$), d'usage (Exhibit A : pas de military, pas de désinformation, etc.).
3. **Vérifier la portée** : la restriction couvre-t-elle seulement les poids, ou aussi les *Outputs* et les *Hosted Services* ? (H3 : les trois.)
4. **Croiser avec son contexte** : juridiction (France = UE), public (ici : dépôt public + écoles = large diffusion), usage (commercial vs éducatif — mais H3 n'exempte pas l'éducatif).
5. **Décider entre** : (a) obtenir une licence commerciale, (b) usage descriptif/analytique sans exécuter ni afficher d'Outputs, (c) choisir une alternative permissive.

Ce notebook incarne l'option **(b)** appliquée à l'enseignement : étudier le modèle sans l'exécuter. C'est honnête (verdict INTRINSIC documenté, pas de contournement) et pédagogiquement riche (la décision de conformité *est* la leçon).


## Exercices

Les trois exercices suivants s'exercent sur la **décision de conformité** — la compétence distinctive de ce notebook. Aucun n'exécute le modèle (la licence l'interdit). Complétez les stubs.


### Exercice 1 — Étendre le vérificateur aux *Hosted Services* (API)

La licence couvre aussi l'usage via API (Art. I.8 *Hosted Services*). Écrivez `can_call_api(country_code, provider_region)` qui renvoie `False` si **soit** le pays de l'appelant, **soit** la région du provider est un territoire exclu — car l'Output serait consommé/affiché dans un territoire exclu.


In [3]:
def can_call_api(country_code: str, provider_region: str) -> bool:
    """True si un appel API MiniMax H3 est autorise pour cet appelant + ce provider.

    Rappel (Art. V.4) : les Outputs ne peuvent etre ni utilises ni affiches hors
    Applicable Territory. Donc l'appel est bloque des que l'appelant OU le provider
    est en territoire exclu.
    """
    # Indice : reutilisez is_applicable_territory(country_code) definie plus haut.
    # Etape 1 : verifier country_code (ou l'appelant consomme l'Output).
    # Etape 2 : verifier provider_region (ou l'Output est produit/affiche).
    # Etape 3 : retourner True seulement si les DEUX sont dans l'Applicable Territory.
    return None  # TODO etudiant


# Test rapide (a decommenter) :
# print(can_call_api("FR", "CN"))   # attendu : False (FR = UE exclue)
# print(can_call_api("JP", "CN"))   # attendu : True (les deux applicables)
# print(can_call_api("JP", "US"))   # attendu : False (US exclue)


### Exercice 2 — Sélectionneur de modèle conforme

Écrivez `choisir_modele_conforme(exigences)` qui, étant donné un dictionnaire d'exigences (pays, `audio_natif` booléen, `open_weights` booléen), renvoie la **liste** des modèles de `MODELES` qui satisfont **toutes** les contraintes — typiquement pour recommander une alternative UE à H3.


In [4]:
def choisir_modele_conforme(exigences: dict) -> list:
    """Renvoie les modeles de MODELES satisfaisant toutes les exigences.

    exigences cles possibles :
      - 'pays' (ISO alpha-2) : le modele doit etre utilisable dans ce pays
      - 'audio_natif' (bool) : le modele doit (ou non) offrir l'audio natif
      - 'open_weights' (bool) : le modele doit (ou non) etre open-weights
    """
    # Indice : filtrez MODELES en cumulatif. Pour 'pays', ecartez les
    # geo-restreints UE si le pays est dans l'UE (is_applicable_territory == False).
    resultats = []
    # TODO etudiant
    return resultats


# Test rapide (a decommenter) :
# besoin = {"pays": "FR", "audio_natif": True, "open_weights": True}
# for m in choisir_modele_conforme(besoin):
#     print(f"  conforme : {m['nom']}")
# # attendu : LTX-2 (UE-ok, audio natif, open-weights). H3 exclu (UE), Sora exclu (pas open-weights).


### Exercice 3 — Détecteur de clause de territorialité dans une licence arbitraire

Écrivez `detecte_restriction_territoriale(texte_licence)` qui prend le texte brut d'une licence et renvoie la liste des noms de pays/régions mentionnés dans une clause d'exclusion géographique (regardez les motifs comme *« excluding »*, *« Excluded Territories »*, *« not permitted in »*). Ceci généralise le raisonnement au-delà de H3.


In [5]:
def detecte_restriction_territoriale(texte_licence: str) -> list:
    """Renvoie les entites geographiques citees dans une clause d'exclusion.

    Recherche des amorces de clause ('excluding', 'Excluded Territories',
    'not permitted in', 'not authorized in') puis extrait les noms de pays/regions
    connus qui suivent.
    """
    entites_connues = [
        "European Union", "United Kingdom", "United States", "United States of America",
        "Republic of Korea", "China", "Japan", "France", "Germany", "Brazil",
    ]
    trouvees = []
    # Indice : pour chaque entite connue, verifier si elle apparait dans le texte
    # A PROXIMITE d'une amorce de clause d'exclusion (pas n'importe ou : une licence
    # peut citer un pays pour d'autres raisons). Une approche simple : chercher
    # l'amorce, puis scanner les N caracteres qui suivent.
    # TODO etudiant
    return trouvees


# Test rapide (a decommenter) avec un extrait de la licence H3 :
# extrait = '''"Excluded Territories" means the European Union, the United Kingdom,
# the Republic of Korea and the United States of America.'''
# print(detecte_restriction_territoriale(extrait))
# # attendu : les 4 territoires exclus


## Conclusion — transparence et conformité comme compétences

MiniMax H3 est techniquement remarquable (#1 video editing, omni-modal, audio natif). Mais **sa licence l'exclut de notre juridiction** (UE), et cette exclusion couvre les poids **comme les Outputs**. Ce notebook a choisi la voie honnête : **documenter et raisonner, pas exécuter ni contourner** (verdict INTRINSIC, [sota-not-workaround](https://github.com/jsboige/CoursIA/blob/main/.claude/rules/sota-not-workaround.md)).

Trois leçons à retenir :

1. **Open-weights ≠ open-source ≠ libre d'usage partout.** Toujours lire la licence à la source et identifier les restrictions structurelles (territoriales, commerciales, d'usage).
2. **La portée compte autant que l'octroi.** Une licence peut autoriser un usage mais en restreindre l'affichage, la redistribution, ou les Outputs — comme H3.
3. **La conformité est une décision de conception, pas une après-pensée.** Choisir tôt entre (a) licence commerciale, (b) usage descriptif, (c) alternative permissive — c'est ce qu'on fait ici avec (b).

**État réversible** : si une licence commerciale UE est obtenue auprès de MiniMax (option A), ce notebook évoluera vers une exécution locale réelle (ComfyUI workflow + VRAM mesurée, comme `02-3-Wan`). En attendant, l'alternative UE pédagogiquement équivalente pour « vidéo + audio natif » est **LTX-2** (`02-5-LTX2-Audiovisual`, licence permissive, exécutable).

---
*Note : ce notebook n'embarque aucun poids, n'appelle aucune API MiniMax, et n'affiche aucun Output généré par H3 — conformément à l'Art. V.4 de la licence. Le code présent analyse le texte de licence et raisonne sur la conformité ; ses sorties sont produites localement par ce raisonnement, pas par le modèle.*
